# Wan 2.1 · High-Fidelity Text to Video on Free Colab (Tesla T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/wan2.1-t4-colab/blob/main/wan2_1_t4_colab_free_inference.ipynb)

A hardware-tailored pipeline running Alibaba's **Wan 2.1 (T2V-1.3B)** with **Native FP16 DiT** and **4-bit UMT5** on a **100% free Google Colab instance** (NVIDIA Tesla T4 GPU, 15 GB VRAM).

**Quick Start:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` *(first run downloads weights ~5–8 min, then cached)*
3. Open the public `gradio.live` link printed in step 4 or generate directly in step 5.

<sub>Architecture: Native FP16 DiT (2.8 GB) · 4-bit NF4 UMT5 (2.4 GB) · UniPC Multistep Solver (22 steps) · Code: <a href="https://github.com/rivas-vega/wan2.1-t4-colab">github.com/rivas-vega/wan2.1-t4-colab</a></sub>

In [ ]:
# @title **1 · Setup** — check GPU and install (~1 min)
import os, sys, subprocess

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return r.stdout

try:
    gpu = _run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).strip()
    print(f"GPU  {gpu}")
except Exception:
    raise SystemExit("No GPU found. Go to Runtime → Change runtime type → T4 GPU, then run again.")

# Prevent Colab Host RAM OOM during safetensors download: disable memory-buffered Xet reconstruction
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

if os.path.isfile("pipeline_utils.py"):
    REPO_DIR = os.getcwd()
elif os.path.isdir("/content/wan2.1-t4-colab"):
    REPO_DIR = "/content/wan2.1-t4-colab"
    try:
        _run(["git", "-C", REPO_DIR, "pull", "-q"])
    except Exception:
        pass
else:
    REPO_DIR = "/content/wan2.1-t4-colab"
    _run(["git", "clone", "-q", "--depth", "1", "https://github.com/rivas-vega/wan2.1-t4-colab.git", REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # Colab's torchao breaks diffusers imports
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "diffusers>=0.33.0", "transformers>=4.48.0", "accelerate>=0.34.0", "bitsandbytes",
      "gradio", "imageio[ffmpeg]", "ftfy", "sentencepiece", "protobuf", "scipy", "psutil"])
print("Ready ✓")

In [ ]:
# @title **2 · Hugging Face token** *(optional — faster download)*
# @markdown Free read token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Leave empty to skip.
HF_TOKEN = ""  # @param {type:"string"}

import os
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or (userdata.get("HF_TOKEN") or "")
except Exception:
    pass

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("Token set ✓  fast authenticated streaming enabled")
else:
    print("No token — using anonymous streaming downloads")

In [ ]:
# @title **3 · Load Wan 2.1 Model** — first run 5–8 min, then cached
import os, warnings, psutil
warnings.filterwarnings("ignore")

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)

from diffusers.utils import logging as _dl
from transformers.utils import logging as _tl
_dl.set_verbosity_error(); _tl.set_verbosity_error()

from pipeline_utils import load_wan_t4_pipeline, clear_cuda_cache

print("Loading Wan 2.1 (Native FP16 DiT + 4-bit UMT5 for 15GB T4 GPU / 12.7GB RAM)…")
pipe = load_wan_t4_pipeline(quantization="fast_fp16")
clear_cuda_cache()

ram_used = (psutil.virtual_memory().total - psutil.virtual_memory().available) / 1e9
ram_total = psutil.virtual_memory().total / 1e9
ram_margin = ram_total - ram_used
vram_used = 0.0
try:
    import torch
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_allocated() / 1e9
except Exception:
    pass

print(f"\nModel Ready ✓ | Host RAM: {ram_used:.1f}/{ram_total:.1f} GB ({ram_margin:.1f} GB safety margin) | VRAM: {vram_used:.2f} GB")

In [ ]:
# @title **4 · Launch Video Studio** — click the public link below
from app import build_demo, launch
demo = build_demo(pipe, output_dir="/content/outputs")
launch(demo, share=True, debug=False)

In [ ]:
# @title **5 · Or generate here** *(optional inline runner)*
PROMPT = "A red vintage sports car drifting around a mountain hairpin at sunset, tire smoke, cinematic tracking shot"  # @param {type:"string"}
PRESET = "Cinematic 480p (HQ)"  # @param ["Cinematic 480p (HQ)", "Widescreen Fast (1.5s)", "Extended Motion (3.0s)"]

from IPython.display import Video, display
from pipeline_utils import create_clip

path, seed, secs, _ = create_clip(pipe, prompt=PROMPT, duration=PRESET, output_dir="/content/outputs")
print(f"Generated in {secs / 60:.1f} min · seed {seed} · {path}")
display(Video(path, embed=True, width=640, html_attributes="controls autoplay loop"))